# Week 6, Day 3 — Context Engineering: Search and Memory
### Local Models Edition — by Abhishek

A trading agent's researcher needs to search the web and remember what it
learns across runs. This lab builds exactly the two servers
`backend/mcp_servers.py`'s `researcher_mcp_servers()` hands to the
researcher — a free web search server and a persistent per-trader memory
server — plus the Fetch server from Day 1.


## The swap: Tavily → free local search

The original course's researcher uses Tavily's hosted MCP server for
search, filtered down to just its `tavily_search` tool — but Tavily needs a
signed-up `TAVILY_API_KEY`. `backend/search_server.py` (already in your
repo) replaces it with `duckduckgo-search`: genuinely free, no signup,
one tool. Have a look at it before running the cells below.


In [ ]:
with open("backend/search_server.py") as f:
    print(f.read())


## 0. Setup

In [ ]:
%pip install -q openai-agents mcp duckduckgo-search python-dotenv


In [ ]:
from openai import AsyncOpenAI
from agents import Agent, Runner, set_default_openai_client, set_tracing_disabled
from agents.mcp import MCPServerStdio

local_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
set_default_openai_client(local_client)
set_tracing_disabled(True)
MODEL_NAME = "llama3.2:3b"

import sys
if sys.platform == "win32":
    import functools, subprocess
    import mcp.client.stdio as mcp_stdio
    mcp_stdio.stdio_client = functools.partial(mcp_stdio.stdio_client, errlog=subprocess.DEVNULL)


## Search on its own

In [ ]:
search_params = {"command": "python", "args": ["-m", "backend.search_server"]}

async with MCPServerStdio(params=search_params, client_session_timeout_seconds=30) as server:
    agent = Agent(
        name="researcher",
        instructions="You research companies using web search.",
        model=MODEL_NAME,
        mcp_servers=[server],
    )
    result = await Runner.run(agent, "Search for what NVIDIA's core business is.")
    print(result.final_output)


## Memory: `mcp-memory-libsql`, exactly as in the real course

`backend/mcp_servers.py` gives each trader's researcher its own persistent
memory database, one SQLite file per name: `memory/{name}.db`. This is
already completely free and local in the original course — no change
needed. Ask it to remember something, then start a *fresh* agent and ask
what it remembers, to see the persistence.


In [ ]:
import os
os.makedirs("memory", exist_ok=True)

memory_params = {
    "command": "npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": "file:./memory/demo.db"},
}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    memory_tools = await server.list_tools()
for t in memory_tools:
    print(t.name, "-", t.description[:80])


In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    agent = Agent(
        name="researcher",
        instructions="You remember useful facts using your memory tools so you don't have to search for them again.",
        model=MODEL_NAME,
        mcp_servers=[server],
    )
    result = await Runner.run(agent, "Remember that NVIDIA's core business is designing GPUs and AI accelerator chips.")
    print(result.final_output)


In [ ]:
# A fresh agent, same memory database - does it remember?
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    agent2 = Agent(
        name="researcher_2",
        instructions="You answer questions using facts you have previously remembered.",
        model=MODEL_NAME,
        mcp_servers=[server],
    )
    result = await Runner.run(agent2, "What do you remember about NVIDIA?")
    print(result.final_output)


## All three together: exactly `researcher_mcp_servers()`

This is the real function from `backend/mcp_servers.py`, called directly —
Fetch, our free search, and this trader's own memory database.


In [ ]:
from backend.mcp_servers import researcher_mcp_servers

servers = researcher_mcp_servers("Warren")
async with servers[0] as fetch, servers[1] as search, servers[2] as memory:
    agent = Agent(
        name="Researcher",
        instructions="You are a financial researcher. Search for news, remember what you learn, and report findings.",
        model=MODEL_NAME,
        mcp_servers=[fetch, search, memory],
    )
    result = await Runner.run(agent, "Find one recent piece of financial news about NVIDIA and remember the key fact.")
    print(result.final_output)


## Recap, and where we are heading

Your researcher now has exactly the toolset it will use on the trading
floor: fetch, free search, and persistent per-trader memory — all free,
all local, all pulled straight from `backend/mcp_servers.py`.

Tomorrow: assembling the trading floor — traders, researchers, and accounts,
working together.

## Exercise
Give two different traders (e.g. "Warren" and "George") their own memory
databases via `researcher_mcp_servers(name)`, have each research and
remember something different, and confirm the databases don't cross-talk —
check the `memory/` folder for the separate `.db` files.
